# Model Specific Trustworthiness Debt in Learning Enabled Systems Using an Ontology

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anonimoustt/IEEESyscon_2027/blob/main/colab_demo.ipynb)

This notebook runs the whole study in about ten minutes:

1. a **static OWL ontology** stores what is known about each model: task, algorithm, parameters, inference time, the ten metrics it watches, and their thresholds;
2. standard **queries** read the information of one model from the ontology (the same files work in Protégé);
3. a **monitor** trains a neural network and a logistic regression on NOAA weather data, sets thresholds from 200, 400, 800, and 1,600 resampled years, validates them on 5,000 new resampled years, and applies them to 40 years of operation. One degraded metric is not enough; two or more related metrics reveal and name a trustworthiness debt.

Run the cells from top to bottom (Runtime, Run all).

## 1. Setup
On Google Colab this cell installs Java (needed by the HermiT reasoner) and the two ontology libraries. Elsewhere it only checks that Java is present.

In [ ]:
import shutil  # find programs on the path
import subprocess  # run shell commands
import sys  # interpreter and loaded modules

IN_COLAB = "google.colab" in sys.modules  # true when running on Google Colab
if IN_COLAB and shutil.which("java") is None:  # Colab without Java
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y default-jre-headless > /dev/null",  # install Java
                   shell=True, check=True)  # stop if the installation fails
if IN_COLAB:  # Colab already has NumPy, pandas, scikit learn, and Matplotlib
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "owlready2==0.48", "rdflib"], check=True)  # ontology libraries
java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()  # Java banner
print([line for line in java if "version" in line][0])  # Java version in use

## 2. Get the repository
Two options:

* **From GitHub:** set `GITHUB_USER` to your GitHub user name. The repository must be public.
* **Without GitHub:** set `GITHUB_USER = ""` (empty). On Colab, an upload button appears; choose `model_specific_trust_debt.zip`.

If the notebook already runs inside the repository, nothing is downloaded.

In [ ]:
import os  # paths
import zipfile  # unpack an uploaded zip

GITHUB_USER = "anonimoustt"  # your GitHub user name, for example "jdoe"
REPO_NAME = "IEEESyscon_2027"  # repository name on GitHub
REPO_DIR = "model_specific_trust_debt"  # local folder name
REPO_URL = "https://github.com/%s/%s.git" % (GITHUB_USER, REPO_NAME)  # address of the public repository
if not os.path.exists("run_all.py"):  # not inside the repository yet
    if not os.path.exists(REPO_DIR):  # not downloaded yet
        if GITHUB_USER not in ("", "YOUR_GITHUB" + "_USERNAME"):  # a real user name was given
            subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)  # clone from GitHub
        elif IN_COLAB:  # no user name: upload the zip instead
            from google.colab import files  # Colab upload widget
            uploaded = files.upload()  # choose model_specific_trust_debt.zip
            zip_name = [n for n in uploaded if n.endswith(".zip")][0]  # the uploaded zip
            zipfile.ZipFile(zip_name).extractall(".")  # unpack it here
        else:  # no user name and no Colab
            raise SystemExit("Set GITHUB_USER, or start the notebook inside the repository folder.")  # stop with a hint
    os.chdir(REPO_DIR)  # work inside the repository
print("working folder:", os.getcwd())  # where the files are

## 3. Run everything
This downloads the weather data (checked by SHA256), builds the ontology, runs all queries, runs the monitor for both models, checks it with an independent implementation, and draws the figure.

In [ ]:
run = subprocess.run([sys.executable, "run_all.py"], capture_output=True, text=True)  # the full pipeline
print(run.stdout)  # progress of the eleven steps (about ten minutes)
print(run.stderr[-2000:])  # last part of any error message

## 4. Get model information from the ontology (SPARQL)
`run_query(name, model)` runs one file from `queries/` exactly as it would run in the Protégé SPARQL Query tab. The example model in each file is `MLP_Weather`; the second argument replaces it.

In [ ]:
sys.path.insert(0, "code")  # make the helper importable
from ask import run_query  # one query, one model, one table
from IPython.display import display  # show tables nicely

print("Q1, MLP_Weather: algorithm, configurations, contexts")  # title
display(run_query("Q1_model_information", "MLP_Weather"))  # neural network
print("Q1, LR_Weather")  # title
display(run_query("Q1_model_information", "LR_Weather"))  # logistic regression

In [ ]:
print("Q2: model information and monitoring settings that the monitor reads")  # title
display(run_query("Q2_monitoring_settings", "MLP_Weather"))  # task, parameters, years, resampling sizes
print("Q3: monitoring plan of LR_Weather (only metrics that apply to logistic regression)")  # title
display(run_query("Q3_monitoring_plan", "LR_Weather"))  # metrics and attribute links
print("Q4: monitored metrics that do not apply to the algorithm")  # title
display(run_query("Q4_inapplicable_metrics"))  # every model
print("Q5: blind spots")  # title
display(run_query("Q5_blind_spots"))  # every model
print("Q6: detectable debts")  # title
display(run_query("Q6_detectable_debts"))  # every model

In [ ]:
print("Q7: attributes and history class of every debt")  # title
display(run_query("Q7_debt_definitions"))  # thirteen rows
WITH_TH = "ontology/trustworthiness_debt_with_thresholds.owl"  # ontology after the thresholds are set
print("Q8 in the static ontology")  # title
display(run_query("Q8_thresholds", "MLP_Weather"))  # empty
print("Q8 after the thresholds are set (averaged over four runs)")  # title
display(run_query("Q8_thresholds", "MLP_Weather", WITH_TH))  # ten threshold pairs

## 5. Case studies, answered by queries alone
ALINA on runways, the AssistTaxi lane detector, and a composite of documented language model failures.

In [ ]:
print("Q9: attributes degraded by documented degradations")  # title
display(run_query("Q9_case_degraded_attributes"))  # nine rows
print("Q10: active debts and their support (two or more confirm a debt)")  # title
display(run_query("Q10_case_active_debts"))  # four rows
print("Q11: complete signatures whose provenance is not on record")  # title
display(run_query("Q11_signatures_without_provenance"))  # ALINA, Capability Loss

## 6. DL queries (HermiT)
These are the Manchester syntax expressions of `queries/dl_queries.txt`. In Protégé, start HermiT, open the DL Query tab, tick Instances, and paste an expression.

In [ ]:
import json  # read results

import pandas as pd  # tables

dl = json.load(open("results/dl_queries.json"))  # answers computed by HermiT in step 3
pd.set_option("display.max_colwidth", None)  # show full expressions
display(pd.DataFrame([{"query": d["query"], "expression": d["manchester"], "instances": ", ".join(d["instances"])}  # one row each
                      for d in dl]))  # all seven

## 7. One metric is not sufficient
Agreement with the independent implementation, the thresholds of each run (Section IV.D), how often each alarm rule fires (Table VI), and the four years discussed in Section IV.E.

In [ ]:
for m in ("MLP_Weather", "LR_Weather"):  # both models
    print(m, "independent check:", json.load(open("results/monitor/cross_check_%s.json" % m)))  # zero mismatches expected

nums = json.load(open("results/paper_numbers.json"))  # numbers of Section IV
n = nums["MLP_Weather"]  # the network
print("Thresholds of MLP_Weather per run (Section IV.D), average, and validation rate (%)")  # title
th = pd.DataFrame(n["thresholds_per_run"])  # one column per run
th["average"] = pd.Series(n["thresholds_avg"])  # averaged threshold
th["validation %"] = pd.Series(n["validation_pct"])  # share of 5,000 validation years beyond it
display(th.round(4))  # Section IV.D
rows = []  # Table VI rows
for m in ("MLP_Weather", "LR_Weather"):  # both models
    n = nums[m]  # numbers of this model
    rows += [{"model": m, "evidence": "accuracy alone", "resampled %": n["validation_pct"]["Accuracy"],  # single metric
              "years (of 40)": n["years_degraded"]["Accuracy"]},  # monitored years
             {"model": m, "evidence": "KS alone", "resampled %": n["validation_pct"]["KS"], "years (of 40)": n["years_degraded"]["KS"]},  # input shift
             {"model": m, "evidence": "any one metric", "resampled %": n["validation_any_one_pct"], "years (of 40)": n["years_any_one"]},  # one
             {"model": m, "evidence": "any two metrics", "resampled %": n["validation_any_two_pct"], "years (of 40)": n["years_any_two"]},  # two
             {"model": m, "evidence": "debt from two related metrics", "resampled %": n["validation_debt_pct"],  # debt rule
              "years (of 40)": n["years_debt"]}]  # monitored years
display(pd.DataFrame(rows))  # Table VI
for m in ("MLP_Weather", "LR_Weather"):  # both models
    print(m, ": accuracy stayed normal in", nums[m]["debt_years_accuracy_normal"], "of", nums[m]["years_debt"], "years with a debt")  # missed

In [ ]:
years = pd.read_csv("results/monitor/monitor_years_MLP_Weather.csv")  # one row per monitored year
cols = ["year", "n_degraded", "deg_Accuracy", "deg_KS", "deg_MMD", "deg_Attack_Success_Rate", "Correctness", "Reliability",  # columns
        "Security", "Resilience", "Robustness_Debt", "Operating_Envelope", "ruled_out"]  # columns, continued
print("The years discussed in Section IV.E of the paper: 17, 21, 31, 37")  # title
display(years[years["year"].isin([17, 21, 31, 37])][cols])  # four of the five points in time

## 8. Figure
Forty monitored years of both models, from raw metrics to attributes to debts.

In [ ]:
from IPython.display import Image  # show a PNG

Image("results/figures/weather.png", width=1000)  # the figure written in step 3

## 9. Next steps

* **Another model:** pass another name to `run_query`, for example `run_query("Q3_monitoring_plan", "MLP_Weather")`.
* **Protégé:** download `ontology/trustworthiness_debt.owl`, open it in Protégé 5.6, and paste any `.rq` file into Window, Tabs, SPARQL Query.
* **A worked example:** `python code/worked_example.py` explains how the thresholds are set and how year 31 is judged.
* **A new model:** add it in `code/build_static_ontology.py` (or in Protégé) and rerun this notebook.